# Nemotron LoRA — train on Colab Pro A100 (4-bit QLoRA)

**Requires Colab Pro + an A100.** On the A100 (sm_80) the fast Mamba kernels compile, so training is ~20-40 min (no slow compile, no torch_forward). 4-bit fits the 40 GB A100.

**First:** Runtime -> Change runtime type -> **A100 GPU**. Run cells top to bottom; the moment training finishes, run the *Save adapter* cell (Colab can drop).

## 0. Confirm an A100

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Code + dependencies (keep Colab's native torch)

In [ ]:
%cd /content
!rm -rf repo && git clone -b build/nemotron-pipeline https://github.com/SebAustin/NVIDIA-Nemotron-Model-Reasoning-Challenge repo
%cd repo
!pip install -q "transformers>=4.45,<5" peft trl datasets accelerate bitsandbytes psutil einops hf_transfer
# we never use vision; a mismatched torchvision breaks transformers' import
!pip uninstall -y -q torchvision torchaudio
import torch
print("TORCH:", torch.__version__, "abi:", torch.compiled_with_cxx11_abi())

## 1b. (Recommended) Cache the model on Google Drive — resumable, download once
Avoids re-downloading the ~63 GB model every session and **resumes** if Colab drops. Needs ~65 GB free Drive (Google One). Set `USE_DRIVE_CACHE=False` to skip.

In [ ]:
USE_DRIVE_CACHE = True
import os
if USE_DRIVE_CACHE:
    from google.colab import drive
    drive.mount('/content/drive')
    os.environ['HF_HOME'] = '/content/drive/MyDrive/hf_cache'
    os.makedirs(os.environ['HF_HOME'], exist_ok=True)
    print('HF cache ->', os.environ['HF_HOME'], '(model persists + resumes across sessions)')

## 2. Build mamba_ssm + causal_conv1d from source (matches Colab's torch, ~10-15 min)

In [ ]:
# Build mamba/causal-conv1d FROM SOURCE against Colab's exact torch (reliable ABI;
# prebuilt wheels often mismatch Colab's torch). ~10-15 min on the A100.
import os
os.environ["CAUSAL_CONV1D_FORCE_BUILD"]="TRUE"; os.environ["MAMBA_FORCE_BUILD"]="TRUE"; os.environ["MAX_JOBS"]="4"
!pip uninstall -y -q mamba-ssm causal-conv1d
!pip install -q ninja packaging wheel setuptools
!pip install -q --no-build-isolation causal-conv1d
!pip install -q --no-build-isolation mamba-ssm
!python -c "import causal_conv1d, mamba_ssm; print('both OK')"

## 3. Hugging Face login (base model may be gated)

In [ ]:
import os
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
from huggingface_hub import login
login()

## 4. Competition data (your Kaggle token)

In [ ]:
import os, glob, shutil, urllib.request
os.makedirs('data', exist_ok=True)
hits = glob.glob('/kaggle/input/**/train.csv', recursive=True)
if hits:
    shutil.copy(hits[0], 'data/train.csv'); print('train.csv (mount) <-', hits[0])
else:
    TOK = "KGAT_xxxxxxxxxxxxxxxx"   # <-- your Kaggle API token (Colab/local path)
    url='https://www.kaggle.com/api/v1/competitions/data/download/nvidia-nemotron-model-reasoning-challenge/train.csv'
    req=urllib.request.Request(url, headers={'Authorization': f'Bearer {TOK}'})
    open('data/train.csv','wb').write(urllib.request.urlopen(req).read())
    print('train.csv (download):', os.path.getsize('data/train.csv'), 'bytes')

## 5. Build the SFT data

In [ ]:
!python scripts/01_eda.py --data-dir data
!python scripts/02_prepare_data.py --data-dir data

## 6. Train (4-bit QLoRA, A100 fast kernels)
Downloads the base model (~63 GB, one-time per session), then trains. Smoke test first. ~20-40 min for 1 epoch on A100; bump NUM_EPOCHS later.

In [ ]:
import os
os.environ['QUANT'] = '4bit'
os.environ['NEMOTRON_MAX_MEMORY_GPU'] = '38GiB'
os.environ['SFT_MAX_SEQ_LENGTH'] = '1024'
os.environ['NUM_EPOCHS'] = '1'
!python scripts/03_train_lora.py --data-path data/train_sft.jsonl --output-dir lora_adapter

## 7. Save the adapter OFF Colab (do this immediately)

In [ ]:
import shutil
shutil.make_archive('/content/lora_adapter', 'zip', 'lora_adapter')
from google.colab import files; files.download('/content/lora_adapter.zip')

## 8. Next: package + submit on Kaggle
Upload `lora_adapter` as a Kaggle dataset, run `kaggle_package_submit.ipynb` -> Save Version -> Submit.